In [ ]:
import pandas as pd
import numpy as np
import glob
import os

def compute_mean_length(path):
    df = pd.read_hdf(path) if path.endswith('.h5') else pd.read_csv(path, header=[0, 1])
    # standardize column levels
    if isinstance(df.columns, pd.MultiIndex):
        nose = df.loc[:,(slice(None), 'nose', 'x' )].values, df.loc[:,(slice(None), 'nose', 'y')].values
        tail = df.loc[:,(slice(None), 'basetail', 'x' )].values, df.loc[:,(slice(None), 'basetail', 'y')].values
    # shape: (n_frames,)
    dist = np.sqrt((nose[0] - tail[0])**2 + (nose[1] - tail[1])**2)
    return dist.mean(), dist.std()

# path to your DLC files
files = glob.glob("/Users/annateruel/Desktop/dlc/*.h5")  # or .csv

results = []
for f in files:
    try:
        mean_len, std_len = compute_mean_length(f)
        results.append({
            'file': os.path.basename(f),
            'mean_nose_tail_px': mean_len,
            'std_px': std_len
        })
    except Exception as e:
        print(f"Failed on {f}: {e}")

df_summary = pd.DataFrame(results)
print(df_summary.sort_values('mean_nose_tail_px'))
with open("/Users/annateruel/Desktop/nose_tail_summary.txt", "w") as f:
    f.write(df_summary.sort_values('mean_nose_tail_px').to_string(index=False))

In [ ]:
import pandas as pd
import numpy as np
import glob
import os

def load_pose_file(path):
    """Load a pose estimation file into a DataFrame."""
    if path.endswith('.h5'):
        return pd.read_hdf(path)
    return pd.read_csv(path, header=[0, 1])

def compute_nose_tail_distance(df):
    """Compute per-frame nose-to-tail Euclidean distances."""
    nose_x = df.loc[:, (slice(None), 'nose', 'x')].values
    nose_y = df.loc[:, (slice(None), 'nose', 'y')].values
    tail_x = df.loc[:, (slice(None), 'basetail', 'x')].values
    tail_y = df.loc[:, (slice(None), 'basetail', 'y')].values
    dists = np.sqrt((nose_x - tail_x)**2 + (nose_y - tail_y)**2)
    return dists

def rescale_dataframe(df, scale_factor, bodyparts):
    """
    Rescale x and y coordinates of specified bodyparts in a DLC-style MultiIndex DataFrame.

    Args:
        df (pd.DataFrame): DataFrame with 3-level MultiIndex columns (scorer, bodypart, coord).
        scale_factor (float): Factor to multiply coordinates by.
        bodyparts (list of str): Bodyparts to rescale (e.g., ['nose', 'head', 'tailbase']).

    Returns:
        pd.DataFrame: Scaled DataFrame.
    """
    df_scaled = df.copy()
    for bp in bodyparts:
        for coord in ['x', 'y']:
            try:
                df_scaled.loc[:, (slice(None), bp, coord)] *= scale_factor
            except KeyError:
                print(f"Warning: Missing coordinate '{coord}' for bodypart '{bp}'")
    return df_scaled

def process_and_rescale_file(path, target_length, outdir, bodyparts):
    """Compute scaling factor and rescale keypoints accordingly."""
    df = load_pose_file(path)
    dists = compute_nose_tail_distance(df)
    mean_len = np.nanmean(dists)
    std_len = np.nanstd(dists)

    scale_factor = target_length / mean_len
    df_scaled = rescale_dataframe(df, scale_factor, bodyparts)

    fname = os.path.splitext(os.path.basename(path))[0] + "_rescaled.h5"
    out_path = os.path.join(outdir, fname)
    df_scaled.to_hdf(out_path, key='df', mode='w')

    return os.path.basename(path), mean_len, std_len, scale_factor, out_path

In [ ]:
files = glob.glob("/Users/annateruel/Desktop/dlc/*.h5")
target_length = 40  # or any other desired length
outdir = "/Users/annateruel/Desktop/dlc/rescaled"
os.makedirs(outdir, exist_ok=True)

summary = []
for f in files:
    try:
        file_name, mean_len, std_len, scale, saved_to = process_and_rescale_file(f, target_length, outdir, bodyparts=['nose', 'rightear', 'leftear', 'head', 'spine1', 'spine2','spine3','basetail'])
        summary.append({
            'file': file_name,
            'mean_nose_tail_px': mean_len,
            'std_px': std_len,
            'scale_factor': scale,
            'saved_to': saved_to
        })
    except Exception as e:
        print(f"Failed on {f}: {e}")

df_summary = pd.DataFrame(summary)
df_summary.sort_values('mean_nose_tail_px', inplace=True)
print(df_summary)

df_summary.to_csv("/Users/annateruel/Desktop/nose_tail_scaling_summary.csv", index=False)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def plot_line_trajectory(df, title="Trajectory", color="red", bp='nose'):
    """
    Plot a line trajectory from a multi-index DataFrame of bodypart coordinates.

    Parameters:
        df (pd.DataFrame): A multi-index DataFrame with (frame, bodypart, coord) columns.
        title (str): Title of the plot.
        color (str): Line color.

    Returns:
        None
    """
    # Extract x and y across all frames and bodyparts
    x = df.loc[:, (slice(None), bp, "x")].values.flatten()
    y = df.loc[:, (slice(None), bp, "y")].values.flatten()

    plt.figure(figsize=(10, 10))
    plt.plot(x, y, marker="o", markersize=2, linestyle="-", color=color)
    plt.gca().invert_yaxis()
    # plt.gca().set_aspect("equal")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(title)
    plt.show()
    
df = pd.read_hdf("/Users/annateruel/Desktop/tesis/behav/interpolated_rescaled/AD22-155-test_bwDLC_DekrW32_srtOct28shuffle1_snapshot_200_filtered.h5_centroid_rescaled.h5")
plot_line_trajectory(df, title="Nose Trajectory", color="blue", bp='head')

In [ ]:
df = pd.read_hdf("/Users/annateruel/Desktop/tesis/behav/interpolated/AD22-155-test_bwDLC_DekrW32_srtOct28shuffle1_snapshot_200_filtered.h5_centroid.h5")
plot_line_trajectory(df, title="Nose Trajectory", color="blue", bp='head')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.collections import LineCollection

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation

def create_tracking_video(df, bodyparts, skeleton, save_path="skeleton_tracking.mp4", fps=30):
    """
    Create a video of skeleton tracking from multi-index dataframe.

    Parameters:
        df (pd.DataFrame): MultiIndex columns (scorer, bodypart, coord).
        bodyparts (list): List of bodyparts to include.
        skeleton (list of tuples): Pairs of bodyparts to connect.
        save_path (str): Path to save the video.
        fps (int): Frames per second.

    Returns:
        None
    """
    n_frames = len(df)
    fig, ax = plt.subplots(figsize=(6, 6))
    scatters = {bp: ax.plot([], [], 'o', label=bp)[0] for bp in bodyparts}
    texts = {bp: ax.text(0, 0, bp, fontsize=8, color='black') for bp in bodyparts}
    lines = [ax.plot([], [], '-', color='gray')[0] for _ in skeleton]

    def init():
        ax.set_xlim(0, df.xs('x', level=2, axis=1).max().max())
        ax.set_ylim(0, df.xs('y', level=2, axis=1).max().max())
        ax.invert_yaxis()
        return list(scatters.values()) + lines + list(texts.values())

    def update(frame):
        for bp in bodyparts:
            x = df.loc[:, (slice(None), bp, 'x')].iloc[frame].values[0]
            y = df.loc[:, (slice(None), bp, 'y')].iloc[frame].values[0]
            scatters[bp].set_data(x, y)
            texts[bp].set_position((x + 5, y))  # Add offset for readability
        for i, (bp1, bp2) in enumerate(skeleton):
            x1 = df.loc[:, (slice(None), bp1, 'x')].iloc[frame].values[0]
            y1 = df.loc[:, (slice(None), bp1, 'y')].iloc[frame].values[0]
            x2 = df.loc[:, (slice(None), bp2, 'x')].iloc[frame].values[0]
            y2 = df.loc[:, (slice(None), bp2, 'y')].iloc[frame].values[0]
            lines[i].set_data([x1, x2], [y1, y2])
        return list(scatters.values()) + lines + list(texts.values())

    ani = animation.FuncAnimation(fig, update, frames=n_frames, init_func=init, blit=True)
    ani.save(save_path, fps=fps, dpi=200)
    plt.close()

In [ ]:
df = pd.read_hdf("/Users/annateruel/Desktop/tesis/behav/interpolated_rescaled/AD22-155-test_bwDLC_DekrW32_srtOct28shuffle1_snapshot_200_filtered.h5_centroid_rescaled.h5")
create_tracking_video(
    df,
    bodyparts=["nose", "rightear", "leftear", "head", "spine1", "basetail"],
    skeleton=[("nose", "head"), ("nose", "rightear"),("nose", "leftear"), ("head", "spine1"), ("spine1", "basetail")],
    save_path="/Users/annateruel/Desktop/example.mp4",
    fps=30
)

In [ ]:
df = pd.read_hdf("/Users/annateruel/Desktop/tesis/behav/interpolated/AD22-155-test_bwDLC_DekrW32_srtOct28shuffle1_snapshot_200_filtered.h5_centroid.h5")
create_tracking_video(
    df,
    bodyparts=["nose", "rightear", "leftear", "head", "spine1", "basetail"],
    skeleton=[("nose", "head"), ("nose", "rightear"),("nose", "leftear"), ("head", "spine1"), ("spine1", "basetail")],
    save_path="/Users/annateruel/Desktop/example_orig.mp4",
    fps=30
)

After adjusting the scale of the bodyparts, we need to adjust also the video, because if not, it will not match the video.

In [ ]:
import pandas as pd
import subprocess
import os
import glob

# --- CONFIGURATION ---
summary_csv = "/Users/annateruel/Desktop/nose_tail_scaling_summary.csv"
video_dir = "/Users/annateruel/Desktop/dlc/"
output_dir = "/Users/annateruel/Desktop/dlc/videos_rescaled"
os.makedirs(output_dir, exist_ok=True)

# --- LOAD CSV ---
df = pd.read_csv(summary_csv)

# --- HELPER TO MATCH VIDEO ---
def find_video_file(h5_name):
    # Remove "_bwDLC..." and file extension
    base = h5_name.split("DLC")[0]
    # Look for matching video
    pattern = os.path.join(video_dir, f"{base}*.mp4")
    matches = glob.glob(pattern)
    return matches[0] if matches else None

# --- LOOP ---
for _, row in df.iterrows():
    h5_name = row["file"]
    scale = row["scale_factor"]

    input_video = find_video_file(h5_name)
    if input_video is None:
        print(f"❌ Video not found for: {h5_name}")
        continue

    base_name = os.path.basename(input_video)
    output_path = os.path.join(output_dir, base_name)
    if os.path.exists(output_path):
        print(f"⏭️  Skipping {base_name} (already exists)")
        continue
    # Get original dimensions
    try:
        probe = subprocess.run(
            [
                "ffprobe", "-v", "error", "-select_streams", "v:0",
                "-show_entries", "stream=width,height", "-of", "csv=p=0", input_video
            ],
            capture_output=True, text=True, check=True
        )
        width, height = map(int, probe.stdout.strip().split(','))
        new_width = int(width * scale)
        new_height = int(height * scale)

        # Make sure both dimensions are divisible by 2
        if new_width % 2 != 0:
            new_width -= 1
        if new_height % 2 != 0:
            new_height -= 1
    except Exception as e:
        print(f"❌ Failed to get resolution for {input_video}: {e}")
        continue

    # Apply ffmpeg scaling
    try:
        result = subprocess.run(
            [
                "ffmpeg", "-y", "-i", input_video,
                "-vf", f"scale={new_width}:{new_height}",
                "-c:a", "copy", output_path
            ],
            capture_output=True, text=True
        )
        if result.returncode != 0:
            print(f"❌ FFmpeg error for {base_name}:\n{result.stderr}")
        else:
            print(f"✅ Scaled {base_name} → {new_width}×{new_height}")
    except Exception as e:
        print(f"❌ Failed to scale {base_name}: {e}")